# Transform Sprints Data
1. Read bronze `sprints` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`constructorId` → `constructor_id`, `driverId` → `driver_id`, `raceName` → `race_name`, `positionText` → `finish_position_text`)
1. Rename columns to make them more meaningful (`date` → `race_date`, `grid` → `grid_position`, `laps` → `completed_laps`, `number` → `car_number`, `position` → `finish_position`)
1. Filter out rows where `season`, `round`, `custructor_id` or `driver_id` is null (business key validation)
1. Remove duplicate records
1. Transform values of column `race_name` to Title Case
1. Write the transformed data to silver `sprints` table


#### Entity Relationship Diagram - Formula1 Bronze Schema

![Formula1 Raw Data.png](../z-course-images/formula1-raw-data-erd.png "Formula1 Raw Data.png")

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"

In [0]:
from pyspark.sql import functions as F

#### Step 1 to 4 Read Source Data, Select required columns & Standardise column names

In [0]:
sprints_df = (
  spark.table(bronze_table)
       .select("season",
              "round",
              "constructorId",
              "driverId",
              "date",
              "raceName",
              "grid",
              "laps",
              "number",
              "points",
              "position",
              "positionText",
              "status",
              "ingestion_timestamp",
              "source_file")
       .withColumnsRenamed({
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "raceName": "race_name",
            "date": "race_date",
            "grid": "grid_position",
            "laps": "completed_laps",
            "number": "car_number",
            "position": "final_position",
            "positionText": "final_position_text"
        })
)

#### Step 5 & 6 Data Quality Checks
- Filter out rows where `season`, `round`, `custructor_id` or `driver_id` is null (business key validation)
- Remove duplicate records

In [0]:
sprints_valid_df = (
    sprints_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull() 
        )
        .dropDuplicates(["season", "round", "constructor_id", "driver_id"])
)

In [0]:
display(sprints_df.count() - sprints_valid_df.count())

#### Step 7 - Transform values of column `nationality` to Title Case

In [0]:
sprints_final_df = (
    sprints_valid_df
        .withColumn('race_name', F.initcap(F.col("race_name")))
)

In [0]:
display(sprints_final_df)

#### Step 8 - Write the transformed data to silver `sprints` table

In [0]:
(
    sprints_final_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))